### Webscraper for extracting newly published names, strains and accession numbers from the weeekly IJSEM email (saved in html). Script then compares the IJSEM names to the NCBI names and generates a report used for taxonomy updates. 
### Uses Natural Language processing combined with REGEX to extract information from text
### Notes on selenium. There is a lot of bad advise in stack overflow that refers to old versions of selenium. Need to refer to the offocial documentation https://selenium-python.readthedocs.io/ If you run out of space in your linux directory the script will crash. There is a cache file written by selenium that may then need to be deleted to get the chrome driver working again. Follow the path in the error message and delete that cache directory. Selenium needs a webdriver installed, find details at https://sites.google.com/chromium.org/driver/downloads

### NOTE selenium cache files are quite large and need to be cleaned up periodically

In [ ]:
pip install "numpy<2.0" 

In [ ]:
pip install spacy
#pip install -U pip setuptools wheel
#pip install -U spacy
#pip install --upgrade spacy

### Install the model at the command lines
small model
python -m spacy download en_core_web_sm

medium model
python -m spacy download en_core_web_md

large model
python -m spacy download en_core_web_lg

### create training set https://spacy.io/usage/training

Label-studio works the best for labelling data. Export as .json when finished, they use labelstudio_to_spacy2.py to convert the .json files to .spacy file for training. 

https://spacy.io/usage/training#quickstart

python -m spacy init fill-config ./base_config.cfg ./config.cfg

This NER annotator worked better than the one above. Used the DocBin technique to convert to .spacy file. Spacy training command with config.cfg then worked.

Config.cfg needs to be created for the spacy command line training. 

Follow the instructions in the quickstart for the base_config.cfg file. Selected OS and clicked NER

https://spacy.io/usage/training#quickstart

Run the following at command line to create the config.cfg. 

python -m spacy init fill-config ./base_config.cfg ./config.cfg

Once the config file was done and spacy file was created run the following at command line to train the model

To debug the data:
python -m spacy debug data config.cfg --paths.train ./train.spacy --paths.dev ./train.spacy

To debug the config file file:
python -m spacy train config.cfg --output ./output

or

python -m spacy train config.cfg 

train the model at command line
python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./train.spacy

creates two directories model-best and model-last. I used model-best 

nlp1 = spacy.load(r"./output/model-best")


Notes
python -m spacy init fill-config ./base_config.cfg ./config.cfg

python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy

https://spacy.io/usage/training#basics

### Train spacy if not already done. Using https://labelstud.io/


In [ ]:
from spacy.tokens import DocBin
import pandas as pd
import json
import os

### train the model at command line
python -m spacy train config3.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy 

### Main webscraper program start


In [1]:
input_file = (r'IJSEMemail89.htm')
output = (r'NameCheckweek89.xlsx')
alldescriptions = (r'all_descriptions89')

In [2]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
import time
import pandas as pd
import re
import os
import sys
import bs4
from bs4 import BeautifulSoup
import numpy as np
from nameparser import HumanName
import spacy

# Base model used for sentence segmentation / general parsing
# nlp = spacy.load("en_core_web_sm")
nlp = spacy.load("en_core_web_md")

# Load your trained NER model ONCE (new training set has multiple entity types)
nlp_strain = spacy.load(r"./output/model-best")  # load once, not inside function

from spacy.matcher import PhraseMatcher
from spacy import displacy  # kept, but not required for extraction

In [3]:
# Debug toggle: set DEBUG_EXTRACT=1 to print what Selenium sees for each URL.
DEBUG_EXTRACT = os.getenv("DEBUG_EXTRACT", "0") == "1"
DEBUG_EXTRACT_LIMIT = int(os.getenv("DEBUG_EXTRACT_LIMIT", "0"))  # 0 means no limit

def _print_page_debug(driver, url):
    print("\nURL:", url)
    print("  current_url:", driver.current_url)
    print("  title:", driver.title)
    print("  page_source_length:", len(driver.page_source or ""))
    title_blocks = driver.find_elements(By.CSS_SELECTOR, "div.tl-main-part.title")
    lowest_blocks = driver.find_elements(By.CLASS_NAME, "tl-lowest-section")
    print("  tl-main-part.title count:", len(title_blocks))
    print("  tl-lowest-section count:", len(lowest_blocks))
    if DEBUG_EXTRACT:
        for i, element in enumerate(title_blocks, start=1):
            txt = (element.text or "").strip()
            if txt:
                print(f"  [title {i}] {txt}")
        for i, element in enumerate(lowest_blocks, start=1):
            txt = (element.text or "").strip()
            if txt:
                print(f"  [lowest {i}] {txt}")



# --- Baseline 1.6 changes: load trained NER model once + organism NER + debug helpers ---
# Trained entity labels: "strain", "accession", "organism", "basionym"

nlp_strain = spacy.load(r"./output/model-best")  # trained model (shared)

ALLOWED_STRAIN_LABELS = {"strain"}
ALLOWED_ORGANISM_LABELS = {"organism"}
ALLOWED_BASIONYM_LABELS = {"basionym"}
ALLOWED_ACCESSION_LABELS = {"accession"}

# PhraseMatcher used to gate strain extraction to sentences mentioning strain keywords
phrase_matcher = PhraseMatcher(nlp.vocab)
strain_keywords = ["strain", "Strain", "strains", "Strains"]
patterns = [nlp.make_doc(p) for p in strain_keywords]
phrase_matcher.add("STRAIN_CTX", patterns)


def _clean_ascii(s: str) -> str:
    return s.encode("ascii", "ignore").decode("utf-8", errors="ignore").strip()


def find_strains(description: str):
    """Return only 'strain' entities from the trained model (sentence-gated)."""
    if not description:
        return []
    results = []
    doc = nlp(description)
    for sent in doc.sents:
        if not phrase_matcher(nlp(sent.text)):
            continue
        doc2 = nlp_strain(sent.text)
        for ent in doc2.ents:
            if ent.label_ in ALLOWED_STRAIN_LABELS:
                val = _clean_ascii(ent.text.strip())
                if val and val not in results:
                    results.append(val)
    return results


def find_organisms(description: str):
    """Return only 'organism' entities from the trained model (no gating)."""
    if not description:
        return []
    results = []
    doc2 = nlp_strain(description)
    for ent in doc2.ents:
        if ent.label_ in ALLOWED_ORGANISM_LABELS:
            val = _clean_ascii(ent.text.strip())
            if val and val not in results:
                results.append(val)
    return results


def find_basionyms(description):
    """
    Find basionym entities using the trained spaCy NER model.
    Returns only entities labeled 'basionym' (de-duplicated).
    """
    if not description:
        return []

    results = []
    doc2 = nlp_strain(description)
    for ent in doc2.ents:
        if ent.label_ in ALLOWED_BASIONYM_LABELS:
            val = ent.text.strip()
            val = val.encode("ascii", "ignore").decode("utf-8", errors="ignore").strip()
            if val and val not in results:
                results.append(val)
    return results


# Debug controls
DEBUG_NER = False
DEBUG_URL_LIMIT = 200  # only debug first N URLs encountered
DEBUG_DESC_PER_URL = 200  # debug only first N descriptions per URL

_debug_seen_urls = set()
_debug_desc_count = {}

# Prefixes we explicitly do NOT want to report (not INSDC sequence accessions)
_NON_INSDC_PREFIXES = (
    "GCA_", "GCF_",  # GenColl/Assembly accessions
    "PRJNA", "PRJEB", "PRJDB",  # BioProject
    "SAMN", "SAME", "SAMD",  # BioSample
)

# INSDC-like accession patterns (sequence records)
_INSDC_REGEXES = [
    re.compile(r"^[A-Z]{1,2}\d{5,8}$", re.I),
    re.compile(r"^[A-Z]{4}\d{8}$", re.I),
    re.compile(r"^[A-Z]{6}\d{9}$", re.I),
    re.compile(r"^[A-Z]{2}_[A-Z]{2}\d{6,}$", re.I),
    re.compile(r"^[A-Z]{2}_[A-Z]{4}\d{8}$", re.I),
]


def find_accessions(description: str):
    """Return only 'accession' entities from the trained model (no gating)."""
    if not description:
        return []
    results = []
    doc2 = nlp_strain(description)
    for ent in doc2.ents:
        if ent.label_ in ALLOWED_ACCESSION_LABELS:
            val = _clean_ascii(ent.text.strip())
            if val and val not in results:
                results.append(val)
    return results


def filter_insdc_accessions(values):
    """Filter accession strings to INSDC-like sequence accessions only.

    Removes BioProject/BioSample/Assembly/GenColl-style IDs and other non-INSDC identifiers.
    Strips version suffixes like '.1'.
    """
    if not values:
        return []
    out = []
    for v in values:
        if v is None:
            continue
        s = str(v).strip()
        if not s or s.lower() == "nan":
            continue
        s = s.rstrip(".,);")
        s = re.sub(r"\.[0-9]+$", "", s)

        if s.upper().startswith(_NON_INSDC_PREFIXES):
            continue

        if any(rx.match(s) for rx in _INSDC_REGEXES):
            if s not in out:
                out.append(s)
    return out


def debug_ner(description: str, *, url: str = "", header: str = ""):
    """Visualize entities from the trained model and show extracted lists."""
    if not DEBUG_NER:
        return
    if url and url not in _debug_seen_urls and len(_debug_seen_urls) >= DEBUG_URL_LIMIT:
        return

    if url:
        _debug_desc_count.setdefault(url, 0)
        if _debug_desc_count[url] >= DEBUG_DESC_PER_URL:
            return
        _debug_desc_count[url] += 1
        _debug_seen_urls.add(url)

    orgs = find_organisms(description)
    strains = find_strains(description)

def remove_non_ascii(text):
    """Remove non-ASCII characters"""
    return ''.join(char for char in text if ord(char) < 128)



### Find URLS from saved email in html - save as from outlook in htm format. URLs are extracted and saved as input for selenium

#### alternative Beautifiul soup code for extracting URLS with autodetect encoding

In [4]:
#alternative Beautifiul soup with autodetect encoding
from charset_normalizer import from_path

# Auto-detect file encoding
result = from_path(input_file).best()
html = str(result)

# Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
text = soup.get_text()

# Extract all http/https links
urls = re.findall(r"https?://\S+", text)
urls = [url.rstrip('.,);') for url in urls]

# Filter links as needed
filtered_urls = [
    url for url in urls
    if all(exclude not in url for exclude in ["TandC", "myaccount", "join-the-society"])
    # Remove "doi.org" from filter if you want article links
]

if not filtered_urls:
    raise ValueError("No usable URLs found!")

print(f"Found {len(filtered_urls)} URLs:")
print("\n".join(filtered_urls))


Found 7 URLs:
http://dx.doi.org/10.1099/ijsem.0.007124
http://dx.doi.org/10.1099/ijsem.0.007109
http://dx.doi.org/10.1099/ijsem.0.007120
http://dx.doi.org/10.1099/ijsem.0.007114
http://dx.doi.org/10.1099/ijsem.0.007123
http://dx.doi.org/10.1099/ijsem.0.007112
http://dx.doi.org/10.1099/ijsem.0.007102


### Main body - Selenium to extract data from html
### https://nameparser.readthedocs.io/en/latest/index.html to find the last name

In [5]:
#Selenium code block
from selenium.webdriver.common.by import By
import tempfile

pub_df = pd.DataFrame(
    columns=['PublishedName', 'Accessions', 'Strains', 'Basionym', 'Authority', 'DOI', 'filtered_url'])
pd.set_option('display.max_columns', None)
combined_description = []

for filtered_url in filtered_urls:
    temp_profile = tempfile.mkdtemp()

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    # Selenium Manager will resolve the matching ChromeDriver/browser automatically.
    driver = webdriver.Firefox(options=options)
    print("Firefox browser version:", driver.capabilities.get("browserVersion"))
    #print("Firefox driver info:", driver.capabilities.get("chrome"))

    counter = 1
    strains = []
    accessions = []
    orgname = []
    doi = []
    author = []
    date = []
    year = []
    author_raw = []
    author = []
    name1 = []
    name2 = []
    author_count = []
    authority = []
    description = None
    description1 = []
    description2 = []
    snumber = []
    basionym = []

    # Navigate to the webpage
    driver.get(filtered_url)

    # Allow time for dynamic content to load (you may need to use WebDriverWait for more robust waiting)
    time.sleep(3)

    _print_page_debug(driver, filtered_url)

    html = driver.execute_script("return document.documentElement.outerHTML")
    soup_live = BeautifulSoup(html, "html.parser")

    title_elements = driver.find_elements(By.CLASS_NAME, "item-meta-data__item-title")
    if DEBUG_EXTRACT:
        print("  item-meta-data__item-title count:", len(title_elements))
    for element in title_elements:
        title = element.text
        print(title)

    doi_elements = driver.find_elements(By.PARTIAL_LINK_TEXT, "doi.org")
    if DEBUG_EXTRACT:
        print("  doi link count:", len(doi_elements))
    for element in doi_elements:
        doi = element.text
        print(doi)

    # find publication year
    for element in driver.find_elements(By.XPATH,
                                        "//*[@id='bellowheadercontainer']/main/div[2]/div/ul/li[3]/span/span[2]"):
        date = element.text
        year = date[-4:]
        print(year)

    # find authors
    for element in driver.find_elements(By.XPATH, "//*[@id='bellowheadercontainer']/main/div[2]/div/ul/li[1]/span"):
        author_raw = element.text
        # print(author_raw)
        author = re.sub(r"[\d+]+", '', author_raw)
        author = re.sub(r"†", '', author)
        author = re.sub(r" and ", ',', author)
        author = re.sub(r",,", ',', author)
        author = author.split(',')
        author[:] = [item for item in author if item != '']
        # print(author)
        author_count = len(author)
        # print(author_count)
        if author_count == 1:
            name1 = (str(author[0]))
            name1 = HumanName(name1)
            authority = name1.last + ' ' + str(year)
            # print(authority)
        elif author_count == 2:
            name1 = (str(author[0]))
            name1 = HumanName(name1)
            name2 = (str(author[1]))
            name2 = HumanName(name2)
            authority = name1.last + ' and ' + name2.last + ' ' + str(year)
            # print(authority)
        else:
            name1 = (str(author[0]))
            name1 = HumanName(name1)
            authority = name1.last + ' et al. ' + str(year)
            # print(authority)

    # extract data from each species description
    for element in driver.find_elements(By.CSS_SELECTOR, "div.tl-main-part.title"):  # finds section headers
        # print(element.text)
        counter += 1
        description = element.text
        print('text', description)
        if "Description of" in description:
            strains = []
            # print('found', description)
            # snumber = 's' + str(counter - 4) + '/p[3]'
            snumber = 's' + str(counter - 4)
            print('snumber is', snumber)
            for element in driver.find_elements(By.ID, snumber):
                description = element.text
                cleaned_text = remove_non_ascii(description)
                combined_description.append(cleaned_text)
                debug_ner(cleaned_text, url=filtered_url, header='Description block (debug)')
                print(cleaned_text)
                print('decription of', description)

                # find the organism names (NER - label 'organism')
                orgname = find_organisms(cleaned_text)

                # find accessions via NER (then post-process to INSDC only)
                accessions = filter_insdc_accessions(find_accessions(cleaned_text))

                # Fallback: regex extraction if NER misses (still filtered to INSDC)
                if not accessions and description is not None:
                    pattern = [r'[A-Z]{2}\d{6}', r'[A-Z]{4}\d{8}', r'([A-Z]+)(_[A-Z]+)\d{6}', r'[A-Z]{6}\d{9}']
                    regex = re.compile(r'\b(' + '|'.join(pattern) + r')\b')
                    accessions = filter_insdc_accessions([m.group() for m in regex.finditer(description)])
                    # print('accessions', accessions)

                # find the strains
                if description is not None:
                    strains = find_strains(cleaned_text)
                    # print('strain names', strains)

                # find the basionyms
                basionym = find_basionyms(cleaned_text) if description is not None else []

                # load data into pandas dataframe
                row_data = [orgname, accessions, strains, basionym, authority, doi, filtered_url]
                length = len(pub_df)
                pub_df.loc[length] = row_data
            print('BREAK1')

    for element in driver.find_elements(By.CLASS_NAME, "tl-lowest-section"):  # finds section headers
        description1 = element.text
        outer_html = element.get_attribute("outerHTML")
        if "Description of" in description1:
            # print(outer_html)
            spans = soup.find_all('span', attrs={'class': 'tl-lowest-section'})
            for span in spans:
                if "Description of" in span.text:
                    # print (span.text)
                    outer_div_id = span.find_parent('div').get('id')
                    # print(f"Outer div ID: {outer_div_id}, Text: {span.text}")
                    for element in driver.find_elements(By.ID, outer_div_id):
                        description = element.text
                        cleaned_text = remove_non_ascii(description)
                        combined_description.append(cleaned_text)
                    debug_ner(cleaned_text, url=filtered_url, header='Description block (debug)')
                    # print(cleaned_text)
                    print(description)

                    # find the organism names (NER - label 'organism')
                    orgname = find_organisms(cleaned_text)

                    # find accessions via NER (then post-process to INSDC only)
                    accessions = filter_insdc_accessions(find_accessions(cleaned_text))

                    # Fallback: regex extraction if NER misses (still filtered to INSDC)
                    if not accessions and description is not None:
                        pattern = [r'[A-Z]{2}\d{6}', r'[A-Z]{4}\d{8}', r'([A-Z]+)(_[A-Z]+)\d{6}', r'[A-Z]{6}\d{9}']
                        regex = re.compile(r'\b(' + '|'.join(pattern) + r')\b')
                        accessions = filter_insdc_accessions([m.group() for m in regex.finditer(description)])
                        # print('accessions', accessions)

                    # find the strains
                    strains = []
                    if description is not None:
                        strains = find_strains(cleaned_text)
                        # print('strain names', strains)

                    # find the basionyms
                    basionym = find_basionyms(cleaned_text) if description is not None else ""

                    basionym_list = find_basionyms(cleaned_text)
                    basionym_for_row = ", ".join(basionym_list) if basionym_list else ""

                    # load data into pandas dataframe
                    row_data = [orgname, accessions, strains, basionym, authority, doi, filtered_url]
                    length = len(pub_df)
                    pub_df.loc[length] = row_data
                    print('BREAK2')

    # Close the browser window
    driver.quit()
print('Scraping Done')

Firefox browser version: 140.10.0

URL: http://dx.doi.org/10.1099/ijsem.0.007124
  current_url: https://www.microbiologyresearch.org/content/journal/ijsem/10.1099/ijsem.0.007124
  title: Three novel diazotrophic species, Paenibacillus oryziterrae sp. nov., Paenibacillus paludis sp. nov. and Paenibacillus diazotrophicus sp. nov., isolated from red paddy soil with potential for consortium-based rice growth promotion | Microbiology Society
  page_source_length: 386732
  tl-main-part.title count: 0
  tl-lowest-section count: 0
Three novel diazotrophic species, Paenibacillus oryziterrae sp. nov., Paenibacillus paludis sp. nov. and Paenibacillus diazotrophicus sp. nov., isolated from red paddy soil with potential for consortium-based rice growth promotion
https://doi.org/10.1099/ijsem.0.007124
2026
Firefox browser version: 140.10.0

URL: http://dx.doi.org/10.1099/ijsem.0.007109
  current_url: https://www.microbiologyresearch.org/content/journal/ijsem/10.1099/ijsem.0.007109
  title: Phreatoba

In [8]:
#optional write description to a file
#print(combined_description)
file = open(alldescriptions, "w")
file.writelines(combined_description)
file.close()

In [9]:
pub_df

,PublishedName,Accessions,Strains,Basionym,Authority,DOI,filtered_url


In [9]:
def non_empty_list(x):
    return isinstance(x, list) and len(x) > 0

pub_df = pub_df[
    pub_df["PublishedName"].apply(non_empty_list) &
    pub_df["Accessions"].apply(non_empty_list)
]

print("Rows after organism + accession filter:", pub_df.shape)

Rows after organism + accession filter: (8, 7)


In [10]:
pub_df

,PublishedName,Accessions,Strains,Basionym,Authority,DOI,filtered_url
0,[Deinococcus pantiae],"[JBJGDW000000000, PP658428]","[SM5_A1T, JCM 36669T, KCTC 43670T]",[],Jiya et al. 2026,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
6,[Halotolerantifilum yawarlongkerapense],"[PX275517, CP199719]","[SD5T, DSM 117693T, ATCC TSD-463T]",[],Fisher et al. 2026,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
10,[Halosubterraneus shenae],"[OR734406, PQ096007, JBMWNX000000000, PP425729...","[AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMC...",[],Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
12,[Pseudomonas stagnisoli],[PV166458],"[P3C3T, CCM 9468T, CECT 31230T, DSM 119661T]",[],Hasan et al. 2026,https://doi.org/10.1099/ijsem.0.007075,http://dx.doi.org/10.1099/ijsem.0.007075
14,[Pseudomonas cimarronensis],[PV166459],"[MAC6T, CCM 9470T, CCUG 78313T, CECT 31231T]",[],Hasan et al. 2026,https://doi.org/10.1099/ijsem.0.007075,http://dx.doi.org/10.1099/ijsem.0.007075
16,[Pectobacterium sinaloense],"[PV590475, SAMN48267889, CP195798]","[LFLA-215T, NCCB 101086T, CMCNRG 1201T, ATCC T...",[],Valdez-López et al. 2026,https://doi.org/10.1099/ijsem.0.007076,http://dx.doi.org/10.1099/ijsem.0.007076
18,[Serinicoccus shuyuelongi],"[PQ579907, PQ579904, JBJFZU000000000, JBJFZT00...","[LYQ92T, GDMCC 1.5391T, KCTC 59547T, LYQ131]",[],Liu et al. 2026,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
20,[Ornithinimicrobium jinqii],"[PQ579903, PQ579902, JBJFZS000000000, JBJFZR00...","[LYQ121T, GDMCC 1.5402T, KCTC 59549T, LYQ103]",[],Liu et al. 2026,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079


In [11]:
pd.set_option('max_colwidth', None)
pub_df = pub_df.copy()
pub_df.loc[:, 'Strains'] = pub_df['Strains'].apply(lambda l: ", ".join(map(str, l)) if isinstance(l, list) else "")
pub_df.loc[:, 'Strains'] = pub_df['Strains'].astype(pd.StringDtype())
pub_df.loc[:, 'Strains'] = pub_df['Strains'].str.replace(',', ', ')
pub_df["Basionym"] = pub_df["Basionym"].apply(lambda x: ", ".join(x) if isinstance(x, list) and len(x) > 0 else "")
pub_df.explode(['PublishedName']).reset_index(drop=True)

,PublishedName,Accessions,Strains,Basionym,Authority,DOI,filtered_url
0,Deinococcus pantiae,"[JBJGDW000000000, PP658428]","SM5_A1T, JCM 36669T, KCTC 43670T",,Jiya et al. 2026,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
1,Halotolerantifilum yawarlongkerapense,"[PX275517, CP199719]","SD5T, DSM 117693T, ATCC TSD-463T",,Fisher et al. 2026,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
2,Halosubterraneus shenae,"[OR734406, PQ096007, JBMWNX000000000, PP425729, PV600743, JBNLVO000000000]","AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",,Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
3,Pseudomonas stagnisoli,[PV166458],"P3C3T, CCM 9468T, CECT 31230T, DSM 119661T",,Hasan et al. 2026,https://doi.org/10.1099/ijsem.0.007075,http://dx.doi.org/10.1099/ijsem.0.007075
4,Pseudomonas cimarronensis,[PV166459],"MAC6T, CCM 9470T, CCUG 78313T, CECT 31231T",,Hasan et al. 2026,https://doi.org/10.1099/ijsem.0.007075,http://dx.doi.org/10.1099/ijsem.0.007075
5,Pectobacterium sinaloense,"[PV590475, SAMN48267889, CP195798]","LFLA-215T, NCCB 101086T, CMCNRG 1201T, ATCC TSD-577",,Valdez-López et al. 2026,https://doi.org/10.1099/ijsem.0.007076,http://dx.doi.org/10.1099/ijsem.0.007076
6,Serinicoccus shuyuelongi,"[PQ579907, PQ579904, JBJFZU000000000, JBJFZT000000000]","LYQ92T, GDMCC 1.5391T, KCTC 59547T, LYQ131",,Liu et al. 2026,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
7,Ornithinimicrobium jinqii,"[PQ579903, PQ579902, JBJFZS000000000, JBJFZR000000000]","LYQ121T, GDMCC 1.5402T, KCTC 59549T, LYQ103",,Liu et al. 2026,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079


In [ ]:
#pub_df['Strains'] = pub_df['Strains'].astype(pd.StringDtype())
#pub_df['Strains'] = pub_df['Strains'].str.replace(',', ', ')
#pub_df

In [12]:
pub2_df = pub_df.explode(['Accessions']).reset_index(drop=True)
#pub2_df
pub4_df = pub2_df.explode(['PublishedName']).reset_index(drop=True)
pub4_df.rename(columns={'Accessions' : 'accession'}, inplace=True)
#pub4_df = pub4_df.dropna()
#pub4_df = pub4_df.drop_duplicates(subset='accession', keep="first")
pub4_df=pub4_df[pub4_df['accession'].isnull() | ~pub4_df[pub4_df['accession'].notnull()].duplicated(subset='accession',keep='first')]
#pub4_df

In [13]:
df_unique= pub4_df.drop_duplicates(["accession"], keep="first")
#df_unique = df_unique.dropna()
df_unique

,PublishedName,accession,Strains,Basionym,Authority,DOI,filtered_url
0,Deinococcus pantiae,JBJGDW000000000,"SM5_A1T, JCM 36669T, KCTC 43670T",,Jiya et al. 2026,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
1,Deinococcus pantiae,PP658428,"SM5_A1T, JCM 36669T, KCTC 43670T",,Jiya et al. 2026,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
2,Halotolerantifilum yawarlongkerapense,PX275517,"SD5T, DSM 117693T, ATCC TSD-463T",,Fisher et al. 2026,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
3,Halotolerantifilum yawarlongkerapense,CP199719,"SD5T, DSM 117693T, ATCC TSD-463T",,Fisher et al. 2026,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
4,Halosubterraneus shenae,OR734406,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",,Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
5,Halosubterraneus shenae,PQ096007,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",,Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
6,Halosubterraneus shenae,JBMWNX000000000,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",,Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
7,Halosubterraneus shenae,PP425729,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",,Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
8,Halosubterraneus shenae,PV600743,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",,Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
9,Halosubterraneus shenae,JBNLVO000000000,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",,Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081


### Create a dataframe of unique accessions and look up NCBI taxonomy information of each accession with srcchk

In [ ]:
#df_unique.dtypes

In [14]:
#df_unique['accession'] = df_unique['accession'].astype('str') 
df_unique.loc[:, 'accession'] = df_unique['accession'].astype('str') 

In [31]:
with open('acclist', 'w') as f:
    for text in df_unique['accession'].tolist():
        f.write(text + '\n')

In [19]:
os.system("/netopt/ncbi_tools64/bin/srcchk -i acclist -f taxname,taxid,strain -o acclist.taxdata")


0

In [15]:
taxdata_file_name = (r'acclist.taxdata')    
srcchk_df = pd.read_csv(taxdata_file_name, sep='\t', index_col=None, low_memory=False)
srcchk_df.drop(columns=['Unnamed: 4'], inplace=True)
srcchk_df.rename(columns={'organism' : 'NCBIname'}, inplace=True)
srcchk_df['accession'] = srcchk_df['accession'].astype(str).replace('\.\d+', '', regex=True).astype(str)
srcchk_df = srcchk_df.dropna(subset=['NCBIname'])
srcchk_df 

,accession,NCBIname,taxid,strain
0,JBJGDW000000000,Deinococcus pantiae,3379094.0,SM5_A1
1,PP658428,Deinococcus sp.,47478.0,SM5_A1
2,PX275517,Eubacteriales bacterium SD5,3459703.0,SD5
3,CP199719,Eubacteriales bacterium SD5,3459703.0,SD5
4,OR734406,Haloparvum sp.,1963352.0,AD34
5,PQ096007,Haloparvum sp.,1963352.0,AD34
6,JBMWNX000000000,Haloparvum sp. AD34,3234950.0,AD34
7,PP425729,Haloparvum sp.,1963352.0,PAK95
8,PV600743,Haloparvum sp.,1963352.0,PAK95
9,JBNLVO000000000,Haloparvum sp. PAK95,3418962.0,PAK95


### Combine dataframes into one

In [16]:
combine_df=pd.merge(left=pub4_df, right=srcchk_df, left_on='accession', right_on='accession', how = 'outer')
combine_df = combine_df[['PublishedName', 'NCBIname', 'Strains', 'accession', 'strain', 'Basionym', 'Authority', 'taxid', 'DOI', 'filtered_url' ]]

# Ensure PublishedName is string for sorting
combine_df['PublishedName'] = combine_df['PublishedName'].astype(str)

# Sort alphabetically (case-insensitive)
combine_df = combine_df.sort_values(
    by='PublishedName',
    key=lambda col: col.str.lower(),
    na_position='last'
).reset_index(drop=True)

combine_df

,PublishedName,NCBIname,Strains,accession,strain,Basionym,Authority,taxid,DOI,filtered_url
0,Deinococcus pantiae,Deinococcus sp.,"SM5_A1T, JCM 36669T, KCTC 43670T",PP658428,SM5_A1,,Jiya et al. 2026,47478.0,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
1,Deinococcus pantiae,Deinococcus pantiae,"SM5_A1T, JCM 36669T, KCTC 43670T",JBJGDW000000000,SM5_A1,,Jiya et al. 2026,3379094.0,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
2,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",PV600743,PAK95,,Ding et al. 2026,1963352.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
3,Halosubterraneus shenae,Haloparvum sp. AD34,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",JBMWNX000000000,AD34,,Ding et al. 2026,3234950.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
4,Halosubterraneus shenae,Haloparvum sp. PAK95,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",JBNLVO000000000,PAK95,,Ding et al. 2026,3418962.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
5,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",OR734406,AD34,,Ding et al. 2026,1963352.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
6,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",PP425729,PAK95,,Ding et al. 2026,1963352.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
7,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",PQ096007,AD34,,Ding et al. 2026,1963352.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
8,Halotolerantifilum yawarlongkerapense,Eubacteriales bacterium SD5,"SD5T, DSM 117693T, ATCC TSD-463T",CP199719,SD5,,Fisher et al. 2026,3459703.0,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
9,Halotolerantifilum yawarlongkerapense,Eubacteriales bacterium SD5,"SD5T, DSM 117693T, ATCC TSD-463T",PX275517,SD5,,Fisher et al. 2026,3459703.0,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084


In [34]:
def highlight_rows(row):
    ijsemvalue = row.loc['PublishedName']
    ncbivalue = row.loc['NCBIname']
    if ijsemvalue != ncbivalue:
        color = '#FFB3BA' # Red
    elif ijsemvalue == ncbivalue:
        color = '#BAFFC9' # Green
    return ['background-color: {}'.format(color) for r in row]

new_df = combine_df.style.apply(highlight_rows, axis=1, subset=['PublishedName', 'NCBIname'])
new_df

,PublishedName,NCBIname,Strains,accession,strain,Basionym,Authority,taxid,DOI,filtered_url
0,Deinococcus pantiae,Deinococcus sp.,"SM5_A1T, JCM 36669T, KCTC 43670T",PP658428,SM5_A1,,Jiya et al. 2026,47478.000000,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
1,Deinococcus pantiae,Deinococcus pantiae,"SM5_A1T, JCM 36669T, KCTC 43670T",JBJGDW000000000,SM5_A1,,Jiya et al. 2026,3379094.000000,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
2,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",PV600743,PAK95,,Ding et al. 2026,1963352.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
3,Halosubterraneus shenae,Haloparvum sp. AD34,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",JBMWNX000000000,AD34,,Ding et al. 2026,3234950.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
4,Halosubterraneus shenae,Haloparvum sp. PAK95,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",JBNLVO000000000,PAK95,,Ding et al. 2026,3418962.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
5,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",OR734406,AD34,,Ding et al. 2026,1963352.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
6,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",PP425729,PAK95,,Ding et al. 2026,1963352.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
7,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",PQ096007,AD34,,Ding et al. 2026,1963352.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
8,Halotolerantifilum yawarlongkerapense,Eubacteriales bacterium SD5,"SD5T, DSM 117693T, ATCC TSD-463T",CP199719,SD5,,Fisher et al. 2026,3459703.000000,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
9,Halotolerantifilum yawarlongkerapense,Eubacteriales bacterium SD5,"SD5T, DSM 117693T, ATCC TSD-463T",PX275517,SD5,,Fisher et al. 2026,3459703.000000,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084


### write output to excel

In [45]:
new_df.to_excel(output, engine='xlsxwriter', index = False, na_rep = '') 

In [ ]:
combine_df.dtypes